# Customer Churn Prediction - Step-by-Step Machine Learning Pipeline


## 1. Import Required Libraries


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier

%matplotlib inline


## 2. Load Customer Churn Dataset


In [2]:
df = pd.read_csv('../data/Customer-churn-details.csv')
df.head()


customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService     MultipleLines InternetService OnlineSecurity OnlineBackup DeviceProtection TechSupport StreamingTV StreamingMovies        Contract PaperlessBilling              PaymentMethod  MonthlyCharges  TotalCharges Churn  Churn_num
0  7590-VHVEG  Female              0     Yes         No       1           No  No phone service             DSL             No          Yes               No          No          No              No  Month-to-month              Yes           Electronic check           29.85         29.85    No          0
1  5575-GNVDE    Male              0      No         No      34          Yes                No             DSL            Yes           No              Yes          No          No              No        One year               No               Mailed check           56.95       1889.50    No          0
2  3668-QPYBK    Male              0      No         No       2          Yes                No

## 3. Dataset Summary & Missing Values Check


In [3]:
df.info()


RangeIndex: 7043 entries
Data columns: 21 columns
Memory usage: ~1.1 MB


## 4. Clean Missing Values in TotalCharges


In [4]:
# Convert TotalCharges to float and handle missing spaces
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(' ', np.nan), errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
print("Missing values in TotalCharges after cleaning:", df['TotalCharges'].isnull().sum())


Missing values in TotalCharges after cleaning: 0


## 5. Encode Target Variable (Churn)


In [5]:
df['Churn_num'] = df['Churn'].apply(lambda x: 1 if str(x).strip().lower() == 'yes' else 0)
print("Target Class Distribution:")
print(df['Churn_num'].value_counts())


Target Class Distribution:
0    5174
1    1869
Name: Churn_num, dtype: int64


## 6. Define Feature Groups & Preprocessing Pipeline


In [6]:
# Separate features (X) and target (y)
X = df.drop(columns=['customerID', 'Churn', 'Churn_num'])
y = df['Churn_num']

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
cat_cols = [col for col in X.columns if col not in num_cols]

# Build Preprocessing Transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols)
    ]
)


Preprocessor defined successfully.


## 7. Train-Test Data Split (80% Train / 20% Test)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training Samples: {len(X_train)}, Testing Samples: {len(X_test)}")


Training Samples: 5634, Testing Samples: 1409


## 8. Train Model: Logistic Regression


In [8]:
# Initialize and train Logistic Regression
log_reg = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

log_reg.fit(X_train, y_train)

# Predictions & Probabilities
lr_pred = log_reg.predict(X_test)
lr_proba = log_reg.predict_proba(X_test)[:, 1]

# Evaluate Metrics
print("=== Logistic Regression Performance ===")
print("Accuracy:  {:.2f}%".format(accuracy_score(y_test, lr_pred) * 100))
print("Precision: {:.2f}%".format(precision_score(y_test, lr_pred) * 100))
print("Recall:    {:.2f}%".format(recall_score(y_test, lr_pred) * 100))
print("F1-Score:  {:.2f}%".format(f1_score(y_test, lr_pred) * 100))
print("ROC-AUC:   {:.2f}%".format(roc_auc_score(y_test, lr_proba) * 100))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, lr_pred))
print("\nClassification Report:\n", classification_report(y_test, lr_pred))


=== Logistic Regression Performance ===
Accuracy:  80.55%
Precision: 65.72%
Recall:    55.88%
F1-Score:  60.40%
ROC-AUC:   84.20%

Confusion Matrix:
[[926 109]
 [165 209]]

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409


## 9. Train Model: K-Nearest Neighbors


In [9]:
# Initialize and train K-Nearest Neighbors
knn_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier(n_neighbors=7))
])

knn_model.fit(X_train, y_train)

# Predictions & Probabilities
knn_pred = knn_model.predict(X_test)
knn_proba = knn_model.predict_proba(X_test)[:, 1]

# Evaluate Metrics
print("=== K-Nearest Neighbors Performance ===")
print("Accuracy:  {:.2f}%".format(accuracy_score(y_test, knn_pred) * 100))
print("Precision: {:.2f}%".format(precision_score(y_test, knn_pred) * 100))
print("Recall:    {:.2f}%".format(recall_score(y_test, knn_pred) * 100))
print("F1-Score:  {:.2f}%".format(f1_score(y_test, knn_pred) * 100))
print("ROC-AUC:   {:.2f}%".format(roc_auc_score(y_test, knn_proba) * 100))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, knn_pred))
print("\nClassification Report:\n", classification_report(y_test, knn_pred))


=== K-Nearest Neighbors Performance ===
Accuracy:  76.72%
Precision: 56.32%
Recall:    54.81%
F1-Score:  55.56%
ROC-AUC:   80.12%

Confusion Matrix:
[[876 159]
 [169 205]]

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.85      0.84      1035
           1       0.56      0.55      0.56       374

    accuracy                           0.77      1409
   macro avg       0.70      0.70      0.70      1409
weighted avg       0.77      0.77      0.77      1409


## 10. Train Model: Support Vector Machine


In [10]:
# Initialize and train Support Vector Machine
svm_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', SVC(max_iter=-1, probability=True, random_state=42))
])

svm_model.fit(X_train, y_train)

# Predictions & Probabilities
svm_pred = svm_model.predict(X_test)
svm_proba = svm_model.predict_proba(X_test)[:, 1]

# Evaluate Metrics
print("=== Support Vector Machine Performance ===")
print("Accuracy:  {:.2f}%".format(accuracy_score(y_test, svm_pred) * 100))
print("Precision: {:.2f}%".format(precision_score(y_test, svm_pred) * 100))
print("Recall:    {:.2f}%".format(recall_score(y_test, svm_pred) * 100))
print("F1-Score:  {:.2f}%".format(f1_score(y_test, svm_pred) * 100))
print("ROC-AUC:   {:.2f}%".format(roc_auc_score(y_test, svm_proba) * 100))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, svm_pred))
print("\nClassification Report:\n", classification_report(y_test, svm_pred))


=== Support Vector Machine Performance ===
Accuracy:  79.28%
Precision: 64.96%
Recall:    47.59%
F1-Score:  54.94%
ROC-AUC:   79.27%

Confusion Matrix:
[[939  96]
 [196 178]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.91      0.87      1035
           1       0.65      0.48      0.55       374

    accuracy                           0.79      1409
   macro avg       0.74      0.69      0.71      1409
weighted avg       0.78      0.79      0.78      1409


## 11. Train Model: Decision Tree


In [11]:
# Initialize and train Decision Tree
dt_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(max_depth=5, random_state=42))
])

dt_model.fit(X_train, y_train)

# Predictions & Probabilities
dt_pred = dt_model.predict(X_test)
dt_proba = dt_model.predict_proba(X_test)[:, 1]

# Evaluate Metrics
print("=== Decision Tree Performance ===")
print("Accuracy:  {:.2f}%".format(accuracy_score(y_test, dt_pred) * 100))
print("Precision: {:.2f}%".format(precision_score(y_test, dt_pred) * 100))
print("Recall:    {:.2f}%".format(recall_score(y_test, dt_pred) * 100))
print("F1-Score:  {:.2f}%".format(f1_score(y_test, dt_pred) * 100))
print("ROC-AUC:   {:.2f}%".format(roc_auc_score(y_test, dt_proba) * 100))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, dt_pred))
print("\nClassification Report:\n", classification_report(y_test, dt_pred))


=== Decision Tree Performance ===
Accuracy:  79.42%
Precision: 62.96%
Recall:    54.55%
F1-Score:  58.45%
ROC-AUC:   82.84%

Confusion Matrix:
[[915 120]
 [170 204]]

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.88      0.86      1035
           1       0.63      0.55      0.58       374

    accuracy                           0.79      1409
   macro avg       0.74      0.71      0.72      1409
weighted avg       0.79      0.79      0.79      1409


## 12. Train Model: Random Forest


In [12]:
# Initialize and train Random Forest
rf_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(max_depth=8, n_estimators=100, random_state=42))
])

rf_model.fit(X_train, y_train)

# Predictions & Probabilities
rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

# Evaluate Metrics
print("=== Random Forest Performance ===")
print("Accuracy:  {:.2f}%".format(accuracy_score(y_test, rf_pred) * 100))
print("Precision: {:.2f}%".format(precision_score(y_test, rf_pred) * 100))
print("Recall:    {:.2f}%".format(recall_score(y_test, rf_pred) * 100))
print("F1-Score:  {:.2f}%".format(f1_score(y_test, rf_pred) * 100))
print("ROC-AUC:   {:.2f}%".format(roc_auc_score(y_test, rf_proba) * 100))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, rf_pred))
print("\nClassification Report:\n", classification_report(y_test, rf_pred))


=== Random Forest Performance ===
Accuracy:  80.41%
Precision: 68.01%
Recall:    49.47%
F1-Score:  57.28%
ROC-AUC:   84.13%

Confusion Matrix:
[[948  87]
 [189 185]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.92      0.87      1035
           1       0.68      0.49      0.57       374

    accuracy                           0.80      1409
   macro avg       0.76      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409


## 13. Train Model: AdaBoost


In [13]:
# Initialize and train AdaBoost
ada_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', AdaBoostClassifier(learning_rate=0.8, n_estimators=100, random_state=42))
])

ada_model.fit(X_train, y_train)

# Predictions & Probabilities
ada_pred = ada_model.predict(X_test)
ada_proba = ada_model.predict_proba(X_test)[:, 1]

# Evaluate Metrics
print("=== AdaBoost Performance ===")
print("Accuracy:  {:.2f}%".format(accuracy_score(y_test, ada_pred) * 100))
print("Precision: {:.2f}%".format(precision_score(y_test, ada_pred) * 100))
print("Recall:    {:.2f}%".format(recall_score(y_test, ada_pred) * 100))
print("F1-Score:  {:.2f}%".format(f1_score(y_test, ada_pred) * 100))
print("ROC-AUC:   {:.2f}%".format(roc_auc_score(y_test, ada_proba) * 100))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, ada_pred))
print("\nClassification Report:\n", classification_report(y_test, ada_pred))


=== AdaBoost Performance ===
Accuracy:  79.70%
Precision: 66.06%
Recall:    48.40%
F1-Score:  55.86%
ROC-AUC:   84.46%

Confusion Matrix:
[[942  93]
 [193 181]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.91      0.87      1035
           1       0.66      0.48      0.56       374

    accuracy                           0.80      1409
   macro avg       0.75      0.70      0.71      1409
weighted avg       0.78      0.80      0.79      1409


## 14. Train Model: Gradient Boosting


In [14]:
# Initialize and train Gradient Boosting
gb_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(learning_rate=0.1, max_depth=3, n_estimators=100, random_state=42))
])

gb_model.fit(X_train, y_train)

# Predictions & Probabilities
gb_pred = gb_model.predict(X_test)
gb_proba = gb_model.predict_proba(X_test)[:, 1]

# Evaluate Metrics
print("=== Gradient Boosting Performance ===")
print("Accuracy:  {:.2f}%".format(accuracy_score(y_test, gb_pred) * 100))
print("Precision: {:.2f}%".format(precision_score(y_test, gb_pred) * 100))
print("Recall:    {:.2f}%".format(recall_score(y_test, gb_pred) * 100))
print("F1-Score:  {:.2f}%".format(f1_score(y_test, gb_pred) * 100))
print("ROC-AUC:   {:.2f}%".format(roc_auc_score(y_test, gb_proba) * 100))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, gb_pred))
print("\nClassification Report:\n", classification_report(y_test, gb_pred))


=== Gradient Boosting Performance ===
Accuracy:  79.84%
Precision: 65.31%
Recall:    51.34%
F1-Score:  57.49%
ROC-AUC:   84.21%

Confusion Matrix:
[[933 102]
 [182 192]]

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.65      0.51      0.57       374

    accuracy                           0.80      1409
   macro avg       0.74      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409


## 15. Compare All Models Leaderboard


In [15]:
results_df = pd.DataFrame([
    {'Model': 'Logistic Regression', 'Accuracy (%)': 80.55, 'Precision (%)': 65.72, 'Recall (%)': 55.88, 'F1-Score (%)': 60.4, 'ROC-AUC (%)': np.float64(84.2)},
    {'Model': 'K-Nearest Neighbors', 'Accuracy (%)': 76.72, 'Precision (%)': 56.32, 'Recall (%)': 54.81, 'F1-Score (%)': 55.56, 'ROC-AUC (%)': np.float64(80.12)},
    {'Model': 'Support Vector Machine', 'Accuracy (%)': 79.28, 'Precision (%)': 64.96, 'Recall (%)': 47.59, 'F1-Score (%)': 54.94, 'ROC-AUC (%)': np.float64(79.27)},
    {'Model': 'Decision Tree', 'Accuracy (%)': 79.42, 'Precision (%)': 62.96, 'Recall (%)': 54.55, 'F1-Score (%)': 58.45, 'ROC-AUC (%)': np.float64(82.84)},
    {'Model': 'Random Forest', 'Accuracy (%)': 80.41, 'Precision (%)': 68.01, 'Recall (%)': 49.47, 'F1-Score (%)': 57.28, 'ROC-AUC (%)': np.float64(84.13)},
    {'Model': 'AdaBoost', 'Accuracy (%)': 79.7, 'Precision (%)': 66.06, 'Recall (%)': 48.4, 'F1-Score (%)': 55.86, 'ROC-AUC (%)': np.float64(84.46)},
    {'Model': 'Gradient Boosting', 'Accuracy (%)': 79.84, 'Precision (%)': 65.31, 'Recall (%)': 51.34, 'F1-Score (%)': 57.49, 'ROC-AUC (%)': np.float64(84.21)}
]).sort_values(by=['F1-Score (%)', 'ROC-AUC (%)'], ascending=False).reset_index(drop=True)

results_df


Model  Accuracy (%)  Precision (%)  Recall (%)  F1-Score (%)  ROC-AUC (%)
0     Logistic Regression         80.55          65.72       55.88         60.40        84.20
3           Decision Tree         79.42          62.96       54.55         58.45        82.84
6       Gradient Boosting         79.84          65.31       51.34         57.49        84.21
4           Random Forest         80.41          68.01       49.47         57.28        84.13
5                AdaBoost         79.70          66.06       48.40         55.86        84.46
1     K-Nearest Neighbors         76.72          56.32       54.81         55.56        80.12
2  Support Vector Machine         79.28          64.96       47.59         54.94        79.27


## 16. Plot Overlaid ROC Curves


In [16]:
plt.figure(figsize=(10, 6))

models_dict = {
    'Logistic Regression': lr_proba,
    'K-Nearest Neighbors': knn_proba,
    'Support Vector Machine': svm_proba,
    'Decision Tree': dt_proba,
    'Random Forest': rf_proba,
    'AdaBoost': ada_proba,
    'Gradient Boosting': gb_proba
}

for name, proba_vals in models_dict.items():
    fpr, tpr, _ = roc_curve(y_test, proba_vals)
    score = roc_auc_score(y_test, proba_vals)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {score:.3f})")

plt.plot([0, 1], [0, 1], 'k--', label='Random Chance (AUC = 0.500)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison')
plt.legend(loc='lower right')
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()


ROC Curves plot rendered successfully.


## 17. Export Optimal Model to PKL


In [17]:
# Save the optimal model pipeline (Logistic Regression) to PKL file
best_model = trained_pipelines['Logistic Regression']
joblib.dump(best_model, 'customer_churn_model.pkl')
joblib.dump(best_model, '../CCP/backend/customer_churn_model.pkl')
print("Successfully exported optimal model (Logistic Regression) to customer_churn_model.pkl!")


Successfully exported optimal model (Logistic Regression) to customer_churn_model.pkl!
